- ### Operations research

1. [Linear programming](#linear-programming)

    1.1. simplex method
    
    1.2. interior-point method

    1.3. the duality principl

2. [Non-linear programming](#non-linear-programming)

    2.1. gradient descent

    2.2. Newton's method

3. [Constrainted programming](#constrained-programming)

    3.1. Lagrange multipliers & KKT condition

4. [Integer programming](#integer-programming)

    4.1. Enumeration
    
    4.2. Branch and bound

5. [Multi-objective optimisation](#multi-objective-optimisation)

    5.1. weighted sum method

    5.2. $\epsilon$-constraint method

    5.3. lexicographic method

6. [Stochastic optimisation](#stochastic-optimisation)

    6.1. sample average approximation

7. [Metaherisitics](#metaheuristics)

    7.1. genetic algorithms
    
    7.2. simulated annealing
    
    7.3. particle swarm optimisation

---

- ### Linear programming

$$\min_x​ ​c^Tx\\ s.t.\ Ax\leq b, x\geq 0.​$$

If one variable $x$ is free, rewrite: 
$$x=z_{1}-z_{2},\quad z_{1},z_{2}≥0.$$

1. simplex method:

move along the edge from feasible vertex

In [13]:
%%capture
%pip install ortools

In [11]:
import numpy as np

def simplex(c, A, b):
    """
    Solve LP:
        maximize c^T x
        subject to Ax <= b, x >= 0
    Using SIMPLEX tableau method.
    """
    m, n = A.shape

    # Build tableau
    tableau = np.zeros((m + 1, n + m + 1))
    tableau[:m, :n] = A
    tableau[:m, n:n + m] = np.eye(m)
    tableau[:m, -1] = b
    tableau[-1, :n] = -c

    while True:
        print(tableau, "\n")
        # Step 1: choose the entering variable (most negative coefficient)
        col = np.argmin(tableau[-1, :-1])
        if tableau[-1, col] >= 0:
            break  # OPTIMAL

        # Step 2: choose leaving variable using minimum ratio test
        ratios = []
        for i in range(m):
            if tableau[i, col] > 0:
                ratios.append(tableau[i, -1] / tableau[i, col])
            else:
                ratios.append(np.inf)
        row = np.argmin(ratios)
        if ratios[row] == np.inf:
            return None, None

        # Step 3: Pivot
        pivot = tableau[row, col]
        tableau[row] /= pivot
        for i in range(m + 1):
            if i != row:
                tableau[i] -= tableau[i, col] * tableau[row]

    # Extract solution
    x = np.zeros(n)
    for j in range(n):
        col = tableau[:, j]
        if np.count_nonzero(col[:-1]) == 1 and np.isclose(col[-1], 0):
            row = np.argmax(col[:-1])
            x[j] = tableau[row, -1]

    z = tableau[-1, -1]
    return x, z


# ========== TEST ==========
c = np.array([-1, 3, -2])
A = np.array([[0, 1, 1], [1, 2, -3], [5, 1, -6]])
b = np.array([4, 2, 1])

x_opt, z_opt = simplex(c, A, b)

print("Optimal x =", x_opt)
print("Optimal Z =", z_opt)


[[ 0.  1.  1.  1.  0.  0.  4.]
 [ 1.  2. -3.  0.  1.  0.  2.]
 [ 5.  1. -6.  0.  0.  1.  1.]
 [ 1. -3.  2.  0.  0.  0.  0.]] 

[[-0.5  0.   2.5  1.  -0.5  0.   3. ]
 [ 0.5  1.  -1.5  0.   0.5  0.   1. ]
 [ 4.5  0.  -4.5  0.  -0.5  1.   0. ]
 [ 2.5  0.  -2.5  0.   1.5  0.   3. ]] 

[[-0.2  0.   1.   0.4 -0.2  0.   1.2]
 [ 0.2  1.   0.   0.6  0.2  0.   2.8]
 [ 3.6  0.   0.   1.8 -1.4  1.   5.4]
 [ 2.   0.   0.   1.   1.   0.   6. ]] 

Optimal x = [0.  2.8 1.2]
Optimal Z = 6.0


In [10]:
import scipy as sp
import numpy as np

"""
    Solve LP:
        minimize c^T x
        subject to Ax <= b, x >= 0
"""

c = np.array([1, -3, 2])
A_ub = np.array([[1, 1, 1], [1, 2, -3], [5, 1, -6]])
b_ub = np.array([4, 2, 1])
res = sp.optimize.linprog(c, A_ub, b_ub)

print("x* = ", res.x)
print("f(x*) = ", res.fun)

x* =  [0.  2.8 1.2]
f(x*) =  -5.999999999999998



2. interior-point method:

$$\min_x ​c^Tx−\mu\sum _{i=1}^m (\log (b_i​−A_i​x) + max(0,-x_i))$$

In [157]:
import numpy as np

def f(vec, A, b, c, mu=10):
    return c@vec - mu*np.sum(np.log(b- A@vec))

def f_g(vec, mu=10):
    return c + mu*sum(A[i]/(b[i]- A[i]@vec) for i in range(A.shape[0]))

def f_j(vec, mu=10):
    return mu*sum(np.outer(A[i],A[i])/((b[i]- A[i]@vec)**2) for i in range(A.shape[0]))

def f_grad(f, vec, h=1e-4):
    return np.array([(f(vec + h*(np.eye(len(vec)))[i], A, b, c)-f(vec, A, b, c))/h for i in range(len(vec))])

def f_J(f_grad, vec, h=1e-4):
    return np.array([[(f_grad(f, vec + h*(np.eye(len(vec)))[i])[j]-f_grad(f, vec)[j])/h for i in range(len(vec))] for j in range(len(vec))])

c = np.array([1, 5])
A = np.array([[1, 1], [1, 3]])
b = np.array([4, 3])
vec = np.array([1, -2])
print(f(vec, A, b, c), "\n")

print(f_g(vec))
print(f_grad(f,vec), "\n")

print(f_J(f_grad,vec))
print(f_j(vec))

-45.88879454113936 

[ 4.25 10.75]
[ 4.25002781 10.75009031] 

[[0.5562697  0.86878913]
 [0.86878913 1.80637088]]
[[0.55625 0.86875]
 [0.86875 1.80625]]


In [47]:
import scipy as sp
import numpy as np

c = np.array([-1, 5])
A = np.array([[1, -1], [-1, -3]])
b = np.array([4, 3])
res = sp.optimize.linprog(c, A, b, method="HiGHS")

print("x* = ", res.x)
print("f(x*) = ", -res.fun)

x* =  [4. 0.]
f(x*) =  4.0



3. the duality principl:

$$\max_x ​c^Tx\\ s.t.\ Ax\leq b, x\geq0.​$$

$$\Leftrightarrow$$

$$\min​ ​b^Ty\\ s.t.\ A^Ty\geq c, y\geq0.​$$

Primal with free variables, dual constraints change:
1. A free variable $x_{j}\rightarrow =$.
2. A constrained variable $x_{j}\ge 0\rightarrow$ j-th inequality $\ge$.
3. A constrained variable $x_{j}\le 0\rightarrow$ j-th inequality $\le$.

In [ ]:
import scipy as sp
import numpy as np

"""
    Solve LP:
        minimize c^T x
        subject to Ax <= b, x >= 0
        aka:
    Solve LP:
        maximize -c^T x
        subject to Ax <= b, x >= 0    
"""

c = np.array([-1, -5])
A = np.array([[1, 1], [1, 3]])
b = np.array([4, 3])
res = sp.optimize.linprog(c, A, b)

print("x* = ", res.x)
print("f(x*) = ", res.fun)

"""
    Solve LP:
        minimize b^T y
        subject to A^Ty >= -c, x >= 0
aka:
    Solve LP:
    miniimize b^T x
    subject to -A^Ty <= c^T, y >= 0
"""

c = b
A_ = -A.T
b_ = c
res_ = sp.optimize.linprog(c, A_, b_)

print("y* = ", res_.x)
print("g(y*) = ", -res_.fun)

x* =  [0. 1.]
f(x*) =  -5.0
y* =  [0.         1.66666667]
g(y*) =  -5.0


---

- ### Non-linear programming

$$\min_{x\in R^n} f(x)$$

1. gradient descent:

$$x_{k+1}=x_k-\alpha_k\nabla f(x_k)$$

In [85]:
import numpy as np

def f(vec):
    return np.sum(vec**2) + np.sum(vec) + 1

def f_grad(f, vec, h=1e-4):
    return [(f(vec + h*(np.eye(vec.shape[0]))[i])-f(vec))/h for i in range(vec.shape[0])]

def gradient_descent(f, vec, alpha=0.01, eps=1e-5, max_iter=1000):
    for i in range(max_iter):
        vec = vec - alpha*np.array(f_grad(f, vec))
        if np.linalg.norm(f_grad(f, vec)) < eps:
            break
    return vec, f(vec)

x, f_x = gradient_descent(f, np.array([-1, 1]))
print("x* = ", x)
print("f(x*) = ", f_x)

x* =  [-0.50005158 -0.50004527]
f(x*) =  0.5000000047095075


2. Newton's method:

$$x_{k+1}=x_k-|\nabla^2 f(x_k)|^{-1} \nabla f(x_k)$$

In [113]:
import numpy as np

def f(vec):
    return np.sum(vec**2) + np.sum(vec) + 1

def f_grad(f, vec, h=1e-4):
    return np.array([(f(vec + h*(np.eye(len(vec)))[i])-f(vec))/h for i in range(len(vec))])

def f_J(f_grad, vec, h=1e-4):
    return np.array([[(f_grad(f, vec + h*(np.eye(len(vec)))[i])[j]-f_grad(f, vec)[j])/h for i in range(len(vec))] for j in range(len(vec))])

def gradient_descent(f_J, f_grad, vec, eps=1e-5, max_iter=1000):
    k = 0
    for i in range(max_iter):
        s = 1/(np.linalg.det(f_J(f_grad, vec)))*f_grad(f, vec)
        vec = vec - s
        k += 1
        if np.linalg.norm(s) < eps:
            break
    return vec, f(vec), k

x, f_x, it = gradient_descent(f_J, f_grad, np.array([-1, 1]))
print("x* = ", x)
print("f(x*) = ", f_x)
print("iterations = ", it)

x* =  [-0.50005191 -0.50004428]
f(x*) =  0.5000000046548734
iterations =  18


---

- ### Constrainted programming

$$\min_{x\in R^n} f(x)$$

$$s.t.\ g_i(x)\leq 0, \\ h_j(x)= 0.$$

1. Lagrange multipliers & KKT condition:

$$min_{x\in R^n} L(x,\lambda_i,\mu_j)=f(x)+\sum_i\lambda_i g_i(x)+\sum_j\mu_j h_j(x)$$
with gradient & dual feasibilty condition:
$$\Rightarrow \left\{\begin{array}{lll}\nabla_{x,\lambda_i,\mu_j} L(x,\lambda_i,\mu_j)=0\\
\lambda_i\cdot g_i(x)=0\\
\lambda_i \geq 0\end{array}\right.$$

In [10]:
import numpy as np
from scipy.optimize import minimize

objective = lambda x: np.sum((x - 3)**2)
constraint_ineq = lambda x: 1 - x # 1 - x_i >= 0 for all i
cons = ({'type': 'ineq', 'fun': constraint_ineq})
initial = np.array([0,0])
solution = minimize(objective, initial, constraints=cons, method='SLSQP')

print(f"Optimal x: {solution.x[0]:.2f}")
print(f"Objective Value: {solution.fun:.2f}")

Optimal x: 1.00
Objective Value: 8.00


---

- ### Integer programming

$$\min_x​ ​c^Tx​$$

$$s.t.\ g_i(x)\leq 0, \\ h_j(x)= 0.$$

1. enumeration:

list all possible lattices

In [236]:
import numpy as np
# minimize: f(x) = c.T*x
#s.t. Ax <= b
#x in {0,1,...,n}

def num_to_digits(k, n, N):
    s = np.zeros(N, dtype=int)
    t = -1
    while k > 0:
        s[t]=k % n
        k //= n
        t -= 1
    return s

def enumerate_search(A, b, c, n):
    y = np.inf
    k = None
    for i in range(n**N):
            x = num_to_digits(i, n, N)
            if np.all(A@x <= b):
                if c@x < y:
                    y = c@x
                    k = x
    return k,y

A = np.array([[17,-20],[-34,20]])
b = np.array([11,2])
c = np.array([3,-5])
n = 5
N = len(c)

k,y = enumerate_search(A, b, c, n)
print("Opitimal value:" ,y)
print("Optimal solution:", k)

Opitimal value: -11
Optimal solution: [3 4]


2. branch and bound:

solve the problem without integer constraints, 

then solve two branches sub-problems if the variable $x_i$ is fractional 
$$x_i\leq\lfloor x_i \rfloor
 \text{  and  } x_i\geq\lceil x_i \rceil$$

In [9]:
import numpy as np
import scipy as sp
import heapq

def solve_lp(c, A, b):
    res = sp.optimize.linprog(c, A_ub=A, b_ub=b)
    return res.x, res.fun

def solve_mip(c, A, b, max_it = 1000):
    task = [([0], A, b)]
    A_, b_ = A, b
    x_opt, fun_opt = None, None
    while task:
        seq, A_, b_ = task[0][0], task[0][1], task[0][2]
        x, fun = solve_lp(c, A_, b_)

        if not (fun == None): # solution exists
            idx = None
            for i in range(len(x)):
                if int_list[i] == 1:
                    if x[i] != np.floor(x[i]):
                        idx = i
            if not idx: # all integer satisfied, update solution
                if not (fun_opt == None):
                    if fun < fun_opt:
                        x_opt, fun_opt = x, fun
                else:
                    x_opt, fun_opt = x, fun
                heapq.heappop(task)
            else: # branch
                x_idx_1 = np.floor(x[idx])
                A_ub_1 = np.zeros((A.shape[0]+1, A.shape[1]))
                A_ub_1[:-1, :] = A
                A_ub_1[-1:, idx] = 1
                
                b_ub_1 = np.zeros((b.shape[0]+1))
                b_ub_1[:-1] = b
                b_ub_1[-1] = x_idx_1
                
                s_1 = seq.copy()
                s_1.append(1)
                heapq.heappop(task)
                if len(seq) < max_it:
                    heapq.heappush(task, (s_1, A_ub_1, b_ub_1))

                x_idx_2 = -np.ceil(x[idx])
                A_ub_2 = np.zeros((A.shape[0]+1, A.shape[1]))
                A_ub_2[:-1, :] = A
                A_ub_2[-1:, idx] = -1
                
                b_ub_2 = np.zeros((b.shape[0]+1))
                b_ub_2[:-1] = b
                b_ub_2[-1] = x_idx_2
                
                s_2 = seq.copy()
                s_2.append(2)
                if len(seq) < max_it:
                    heapq.heappush(task, (s_2, A_ub_2, b_ub_2))

        else:
            heapq.heappop(task)
    return x_opt, fun_opt

c = np.array([1, -3, 2])
A = np.array([[0, 1, 1], [1, 2, -3], [5, 1, -6]])
b = np.array([4, 2, 1])
int_list = np.array([1, 0, 1])

x_opt, fun_opt = solve_mip(c, A, b)
print("x_opt: ", x_opt, "fun_opt: ", fun_opt)

x_opt:  [0.  2.5 1. ] fun_opt:  -5.5


In [6]:
import numpy as np
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver("CBC")
int_list = np.array([1, 0, 1])
n = len(int_list)

# Continuous vars
x = [solver.NumVar(0, solver.infinity(), f"x[{i}]") for i in range(n)]

# Integer vars
for i in range(n):
    if int_list[i] == 1:
        x[i].SetInteger(True)

c = np.array([1, -3, 2])
A = np.array([[0, 1, 1], [1, 2, -3], [5, 1, -6]])
b = np.array([4, 2, 1])

for i in range(len(A)):
    solver.Add(sum(A[i][j] * x[j] for j in range(len(x))) <= b[i])

solver.Minimize(sum(c[j] * x[j] for j in range(len(x))))

status = solver.Solve()
if status == pywraplp.Solver.OPTIMAL:
    solver.Solve()
    s = [x[j].solution_value() for j in range(n)]
    print(s)
    print("Objective:", solver.Objective().Value())

[0.0, 2.5, 1.0]
Objective: -5.5


---

- ### Multi-objective optimisation


$$\min​_x ​F(x)=[f_1(x),f_2(x),\dots,f_k(x)]^T​$$

$$s.t.\ g_i(x)\leq 0, \\ h_j(x)= 0.$$

1. weighted sum method:


$$\min_x​ \sum_{i=1}^kw_if_i(x)$$

where

$$\sum_{i=1}^kw_i=1,w_i\geq0$$

2. $\epsilon$-constraint method

$$\min​_x ​f_1(x)$$

$$s.t.\ ​f_2(x)\leq\epsilon_2,\\\dots,\\​f_k(x)\leq\epsilon_k,
\\g_i(x)\leq 0, \\ h_j(x)= 0.$$

3. lexicographic method


optimize $f_1$ first, then among all optimal $f_1$ optimize $f_2$, and then continue until $f_k$

---

- ### Stochastic optimisation

$$\min_x \mathbb{E}_{\xi}[f(x,\xi)]$$
$$s.t.\ \Pr {\{g_i(x,\xi)\leq0\}}\geq1-\alpha$$

1. sample average approximation:

$$\min_x \frac{1}{N}\sum_{i=1}^Nf(x,\xi_i)$$
$$s.t.\ \frac{1}{N}\sum_{j=1}^N1\{g_i(x,\xi_j)\}\geq1-\alpha$$


---

- ### Metaherisitics

$$x^{(t+1)}=MetaHeuristicStep(x^{(t)},f,randomness)$$

**genetic algorithm:**

In [ ]:
import numpy as np
from sympy import latex, symbols
from IPython.display import display, Math

def geneticalgorithm(f,x_min,x_max,n_gens,pop_size,mutation_rate=0.01):
    # Initialize population
    pop = np.random.uniform(x_min, x_max, (pop_size//2*2, len(x_min)))
    # Evolution process
    for gen in range(n_gens):
        next_pop = np.empty((0, len(x_min)))
        for _ in range(pop_size // 2):
            # Selection (tournament selection)
            def selection(pop):
                idx1, idx2 = np.random.randint(0, len(pop), 2)
                return pop[idx1] if f(pop[idx1]) < f(pop[idx2]) else pop[idx2]
            parent1 = selection(pop)
            parent2 = selection(pop)

            # Crossover
            crossover_point = np.random.randint(1, len(x_min)-1)
            child1 = np.concatenate((parent1[:crossover_point], parent2[crossover_point:]))
            child2 = np.concatenate((parent2[:crossover_point], parent1[crossover_point:]))
            next_pop = np.vstack((next_pop, child1)) 
            next_pop = np.vstack((next_pop, child2))

        # Mutation
        for i in range(pop_size):
            for j in range(len(x_min)):
                if np.random.rand() < mutation_rate:
                    next_pop[i][j] += (x_max[j] - x_min[j]) * (n_gens - gen) / n_gens * np.random.normal(0, 1)
        pop = next_pop
    return pop[np.argmin([f(ind) for ind in pop])]

def f(t): # objective function to minimize
    global expr
    global variables
    variables = symbols('x1:' + str(len(t)+1)) # define symbolic variables x1:n
    expr = variables[0]**2 + variables[1]**2 + variables[2]**2 + 1
    return expr.subs(subs_dict(variables, t))

def subs_dict(variables, t): # create substitution dictionary
    return {variables[i]: t[i] for i in range(len(variables))}

x_min = np.array([-1,-1,-1])
x_max = np.array([1,1,1])
x_opt = geneticalgorithm(f,x_min,x_max,100,100)
display(Math(r"f(x)="+latex(expr)))
print("Minimal solution:\n x* =", x_opt, "\nf(x*) =", f(x_opt))

<IPython.core.display.Math object>

Minimal solution:
 x* = [-0.00315474  0.00138191  0.00787225] 
f(x*) = 1.00007383439643


**simulated annealing:**

In [4]:
import numpy as np
import math
from sympy import latex, symbols
from IPython.display import display, Math

def simulated_annealing(f,x,T0,kmax):
    for k in range(kmax-1):   
        T = T0*(1-(k+1)/kmax)    # cooling schedule
        x_ = x + T * np.random.randn(len(inital_value))  # generate neighbor of x 
        if f(x_) - f(x) <= 0: # accept new solution if better
            x = x_
        elif np.random.uniform(0,1) <= math.exp((f(x) - f(x_)) / T): # accept with certain probability
            x = x_
    return x

def f(t): # objective function to minimize
    global expr
    global variables
    variables = symbols('x1:' + str(len(t)+1)) # define symbolic variables x1:n
    expr = variables[0]**2 + variables[1]**2 + variables[2]**2 + 1
    return expr.subs(subs_dict(variables, t))

def subs_dict(variables, t): # create substitution dictionary
    return {variables[i]: t[i] for i in range(len(variables))}

inital_value = np.array([0.5,0.5,0.5])
x_opt = simulated_annealing(f,inital_value,1,1000)
display(Math(r"f(x)="+latex(expr)))
print("Minimal solution:\n x* =", x_opt, "\nf(x*) =", f(x_opt))

<IPython.core.display.Math object>

Minimal solution:
 x* = [-0.17549098 -0.09413973  0.0681766 ] 
f(x*) = 1.04430742251557


**particle swarm optimisation:**

In [24]:
import numpy as np
from sympy import latex, symbols
from IPython.display import display, Math

def particle_swarm(f,x_min,x_max,n_particles,kmax,w=0.5,c1=1,c2=1):
    particles = np.random.uniform(x_min, x_max, (n_particles, len(x_min)))
    velocities = np.random.uniform(-abs(x_max - x_min), abs(x_max - x_min), (n_particles, len(x_min)))
    person_best = particles.copy()
    global_best = particles[np.argmin([f(p) for p in particles])]

    # Optimization loop
    for k in range(kmax):
        for i in range(n_particles):
            r1, r2 = np.random.rand(), np.random.rand()
            velocities[i] = (w * velocities[i] +
                             c1 * r1 * (person_best[i] - particles[i]) +
                             c2 * r2 * (global_best - particles[i]))
            particles[i] += velocities[i]

            # Update personal best
            if f(particles[i]) < f(person_best[i]):
                person_best[i] = particles[i]

        # Update global best
        current_global_best = particles[np.argmin([f(p) for p in particles])]
        if f(current_global_best) < f(global_best):
            global_best = current_global_best
    return global_best
def f(t): # objective function to minimize
    global expr
    global variables
    variables = symbols('x1:' + str(len(t)+1)) # define symbolic variables x1:n
    expr = variables[0]**2 + variables[1]**2 + variables[2]**2 + 1
    return expr.subs(subs_dict(variables, t))

def subs_dict(variables, t): # create substitution dictionary
    return {variables[i]: t[i] for i in range(len(variables))}

x_min = np.array([-1,-1,-1])
x_max = np.array([1,1,1])
x_opt = particle_swarm(f,x_min,x_max,10,10)
display(Math(r"f(x)="+latex(expr)))
print("Minimal solution:\n x* =", x_opt, "\nf(x*) =", f(x_opt))

<IPython.core.display.Math object>

Minimal solution:
 x* = [-0.00283797  0.03078284  0.00498104] 
f(x*) = 1.00098044776241
